In [1]:
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision import datasets
from skimage.feature import local_binary_pattern

class our_preprocessing:
    def __init__(self, size, threshold, crop, method):
        self.size = size
        self.threshold = threshold
        self.crop = crop
        self.method = method  # 'Fixed', 'Mean', 'Median', 'Otsu', 'Adaptive'

    def __call__(self, img):
        # 1. PIL → NumPy (RGB) & Crop & Grayscale
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h - self.crop, self.crop:w - self.crop, :]
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # 3. Edge extraction(LoG)
        log_edge = np.abs(cv2.Laplacian(gray, cv2.CV_64F, ksize=3)) 
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3);sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_edge = np.sqrt(sobelx ** 2 + sobely ** 2)
         # 먼저 정규화 (0~1)
        log_norm = cv2.normalize(log_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        sobel_norm = cv2.normalize(sobel_edge, None, 0, 1.0, cv2.NORM_MINMAX)
         # 자동 가중치: 평균 강도에 비례해서 반영 (더 강한 에지에 더 가중치)
        log_weight = np.mean(log_norm)
        sobel_weight = np.mean(sobel_norm)
        total = log_weight + sobel_weight
        w1 = log_weight / total
        w2 = sobel_weight / total
         # 가중 평균
        edge = w1 * log_norm + w2 * sobel_norm
        edge = cv2.normalize(edge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8) # 감마 안 쓸 때만 활성화 
 
        # 4. Gaussian Blur & Resize( resize 필수 (프로젝트 가이드라인)) 
        blurred = cv2.GaussianBlur(edge, (3, 3), 1)
        resized = cv2.resize(blurred, (self.size, self.size), interpolation=cv2.INTER_CUBIC)
        resized = resized.astype(np.uint8) 
        
        # 5. Binarization by method
        if self.method == 'Fixed':
            _, binarized = cv2.threshold(resized, self.threshold, 255, cv2.THRESH_BINARY)
        elif self.method == 'Otsu':
            _, binarized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        elif self.method == 'Adaptive':
            binarized = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                              cv2.THRESH_BINARY, 11, 2)
        else:
            raise ValueError(f"Unknown binarization method: {self.method}")
            
        # 6. NumPy → PIL → Tensor
        final_img = Image.fromarray(binarized)
        return transforms.functional.to_tensor(final_img)

In [2]:
import torch
size_transform = transforms.Compose([
    our_preprocessing(size=64, threshold=128, crop=2, method='Otsu')
])

# Dataset 정의
trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train', transform=size_transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True)

testset  = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test',  transform=size_transform)
testloader = torch.utils.data.DataLoader(testset, batch_size = 40, shuffle=False, num_workers=2, drop_last=True)  

# of trainset = 788
# of trainloader = 200
input size: torch.Size([1, 64, 64]), 2


In [3]:
import numpy as np 
import torch.optim as optim    
from tqdm import tqdm 

def train(model, device, trainloader, optimizer, criterion, num_epochs):
    history = np.zeros((0, 3))  # 각 행: [epoch, avg_loss, avg_accuracy]
    
    for epoch in tqdm(range(num_epochs)):
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0

        for X, y in trainloader: 
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            predict = model(X)
            loss = criterion(predict, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pred_class = predict.argmax(dim=1)
            correct += (pred_class == y).sum().item()
            total += y.size(0)

        avg_loss = epoch_loss / len(trainloader)
        avg_accuracy = correct / total

        item = np.array([epoch + 1, avg_loss, avg_accuracy])
        history = np.vstack((history, item))

        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.6f}, Accuracy: {avg_accuracy:.4f}")
        
    return history
    
def test(model, device, test_loader, criterion):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for X, y in tqdm(test_loader): 
            X = X.to(device)
            y = y.to(device)

            predict = model(X)
            loss = criterion(predict, y)

            pred_class = predict.argmax(dim=1)
            accuracy = (pred_class == y).float().mean()

            test_accuracy.append(accuracy.item())
            test_loss.append(loss.item())

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f'test loss : {avg_loss:.4f} / test_accuracy : {avg_accuracy:.4f}')

In [13]:
## resnet model ##
import torchvision.models as models
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
prob1_3 = models.resnet18()
num_ftrs = prob1_3.fc.in_features
prob1_3.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
prob1_3.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4),
)

prob1_3 = prob1_3.to(device)
# 하이퍼파라미터 설정
num_epochs = 47
lr = 0.001
optimizer = torch.optim.Adam(prob1_3.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss()

# 학습 실행
history = train(prob1_3, device, trainloader, optimizer, criterion, num_epochs)
result = test(prob1_3, device, testloader, criterion)

  2%|▉                                           | 1/47 [00:18<13:55, 18.16s/it]

Epoch [1/47] - Loss: 0.987279, Accuracy: 0.5721


  4%|█▊                                          | 2/47 [00:35<13:13, 17.64s/it]

Epoch [2/47] - Loss: 0.781138, Accuracy: 0.6806


  6%|██▊                                         | 3/47 [00:54<13:20, 18.20s/it]

...

Epoch [46/47] - Loss: 0.044837, Accuracy: 0.9844


100%|███████████████████████████████████████████| 47/47 [13:53<00:00, 17.74s/it]


Epoch [47/47] - Loss: 0.012814, Accuracy: 0.9961


100%|███████████████████████████████████████████| 19/19 [00:01<00:00,  9.50it/s]

test loss : 1.5294 / test_accuracy : 0.7263
